# Telco Customer Churn — Exploratory Data Analysis

**Dataset:** `telco_churn_cleaned.csv` (output of preprocessing notebook)  
**Rows:** 7,043 | **Columns:** 20 (customerID dropped, Churn encoded as 0/1)  
**Goal:** Understand what drives customer churn through visual and statistical exploration.

> **Note:** This notebook assumes the cleaned CSV from the preprocessing notebook is in the same directory. If you're starting fresh, replace the load path with the raw CSV and re-apply the cleaning steps first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Global style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

CHURN_COLORS = {0: '#4C72B0', 1: '#DD4949'}   # blue = no churn, red = churn
BAR_COLOR    = '#DD4949'
BASE_COLOR   = '#4C72B0'

print('Setup complete.')

In [ ]:
df = pd.read_csv('telco_churn_cleaned.csv')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Helper: annotate bar charts with percentage labels
def annotate_bars(ax, fmt='{:.1f}%', offset=0.5):
    for bar in ax.patches:
        h = bar.get_height()
        if h > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                h + offset,
                fmt.format(h),
                ha='center', va='bottom', fontsize=9, fontweight='bold'
            )

# Helper: compute churn rate (%) grouped by one column
def churn_rate(col):
    return df.groupby(col)['churn'].mean().mul(100).reset_index().rename(columns={'churn': 'churn_rate'})

---
## Section 1 — Overall Churn Snapshot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 1a. Absolute counts
counts = df['churn'].value_counts().rename({0: 'No Churn', 1: 'Churned'})
axes[0].bar(counts.index, counts.values,
            color=[BASE_COLOR, BAR_COLOR], edgecolor='white', width=0.5)
axes[0].set_title('Churn Count')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 40, f'{v:,}', ha='center', fontsize=11, fontweight='bold')

# 1b. Pie
pct = df['churn'].value_counts(normalize=True) * 100
axes[1].pie(
    pct.values,
    labels=['No Churn', 'Churned'],
    colors=[BASE_COLOR, BAR_COLOR],
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2),
    textprops=dict(fontsize=11)
)
axes[1].set_title('Churn Proportion')

fig.suptitle('Overall Churn Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Overall churn rate: {df["churn"].mean()*100:.1f}%')
print(f'Total customers: {len(df):,} | Churned: {df["churn"].sum():,} | Retained: {(df["churn"]==0).sum():,}')

**Caption:** About 1 in 4 customers churns. The dataset is moderately imbalanced — worth keeping in mind during modeling.

---
## Section 2 — Churn Across Demographics

In [ ]:
demo_cols = ['gender', 'senior_citizen', 'partner', 'dependents']
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, col in zip(axes, demo_cols):
    cr = churn_rate(col).sort_values('churn_rate', ascending=False)
    bars = ax.bar(cr[col].astype(str), cr['churn_rate'],
                  color=BAR_COLOR, edgecolor='white', width=0.5, alpha=0.88)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, 65)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    annotate_bars(ax, offset=0.8)

fig.suptitle('Churn Rate by Demographic Feature', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Caption:**
- **Gender** has virtually no effect — Male and Female churn at almost identical rates (~26%). Gender is not a useful predictor.
- **Senior Citizens** churn at ~41%, nearly double the rate of non-seniors (~24%). A high-risk segment.
- **Partner** acts as an anchor — customers without a partner churn at ~33% vs ~20% for those with one.
- **Dependents** follow the same pattern — customers with dependents churn at ~15%, those without at ~31%. Family ties reduce churn significantly.

In [ ]:
# Stacked bar: full population breakdown — churned vs retained per demographic group
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, col in zip(axes, demo_cols):
    ct = df.groupby([col, 'churn']).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.rename(columns={0: 'Retained', 1: 'Churned'}).plot(
        kind='bar', stacked=True, ax=ax,
        color=[BASE_COLOR, BAR_COLOR], edgecolor='white', width=0.5
    )
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Percentage (%)')
    ax.set_ylim(0, 110)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.legend(loc='upper right', fontsize=8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

fig.suptitle('Retained vs Churned — Demographic Breakdown (100% Stacked)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Caption:** The stacked view makes it visually clear how dominant the churn difference is for Senior Citizens and the family-anchored variables (Partner, Dependents). Gender bars look nearly identical — confirming it adds no signal.

---
## Section 3 — Churn vs Services

In [ ]:
service_cols = [
    'internet_service', 'tech_support', 'online_security',
    'online_backup', 'device_protection', 'streaming_tv', 'streaming_movies'
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for ax, col in zip(axes, service_cols):
    cr = churn_rate(col).sort_values('churn_rate', ascending=False)
    bar_colors = [BAR_COLOR if v > 26 else BASE_COLOR for v in cr['churn_rate']]
    ax.bar(cr[col].astype(str), cr['churn_rate'],
           color=bar_colors, edgecolor='white', width=0.55, alpha=0.9)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, 75)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.axhline(df['churn'].mean() * 100, color='black',
               linestyle='--', linewidth=1, label='Overall avg')
    ax.legend(fontsize=7)
    annotate_bars(ax, offset=0.8)
    ax.tick_params(axis='x', rotation=15)

# Hide unused subplot
axes[-1].set_visible(False)

fig.suptitle('Churn Rate by Service Type\n(Dashed line = overall average 26.5%)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Caption:**
- **Internet Service:** Fiber optic customers churn at ~42% — the single highest churn rate of any service category. DSL customers churn at ~19%. Customers with no internet churn at only ~7%.
- **Tech Support & Online Security:** Customers without these add-ons churn at 2x the rate of those who have them. These services appear to create stickiness.
- **Streaming TV & Movies:** Minimal differentiation — churn rate is roughly the same whether subscribed or not. These are not retention drivers.
- **Online Backup & Device Protection:** Moderate effect — subscribed customers churn slightly less, but the gap is smaller than Security/Support.

In [ ]:
# Heatmap: churn rate by InternetService × TechSupport
pivot = df[df['internet_service'] != 'No'].pivot_table(
    index='internet_service', columns='tech_support',
    values='churn', aggfunc='mean'
) * 100

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r',
            vmin=0, vmax=60, linewidths=0.5, ax=ax,
            annot_kws={'size': 12, 'weight': 'bold'})
ax.set_title('Churn Rate (%) — Internet Service × Tech Support', pad=12)
ax.set_xlabel('Tech Support')
ax.set_ylabel('Internet Service')
plt.tight_layout()
plt.show()

**Caption:** Fiber optic customers without Tech Support churn at ~50% — half of them leave. Adding Tech Support drops that to ~29%. For DSL, the effect is similarly significant (28% → 12%). This interaction is one of the most actionable findings in the dataset.

---
## Section 4 — Churn vs Tenure, Monthly Charges, and Contract Type

In [ ]:
# 4.1 Boxplots: tenure and monthly_charges by churn
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, title in zip(
    axes,
    ['tenure', 'monthly_charges'],
    ['Tenure (months)', 'Monthly Charges ($)']
):
    groups = [df[df['churn'] == 0][col], df[df['churn'] == 1][col]]
    bp = ax.boxplot(
        groups, labels=['No Churn', 'Churned'],
        patch_artist=True, notch=True,
        medianprops=dict(color='black', linewidth=2),
        whiskerprops=dict(color='gray'),
        capprops=dict(color='gray'),
        flierprops=dict(marker='o', color='gray', alpha=0.2, markersize=3)
    )
    for patch, color in zip(bp['boxes'], [BASE_COLOR, BAR_COLOR]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_title(f'{title} by Churn Status')
    ax.set_ylabel(title)

    # Annotate medians
    for i, grp in enumerate(groups, 1):
        med = grp.median()
        ax.text(i, med + 1, f'Median: {med:.0f}', ha='center', fontsize=9, color='black')

plt.suptitle('Tenure & Monthly Charges Distribution by Churn Status', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Caption:**
- **Tenure:** Churned customers have a median tenure of ~10 months; retained customers ~38 months. Churn is heavily front-loaded — most exits happen early in the customer lifecycle.
- **Monthly Charges:** Churned customers pay more on average (median ~$79 vs ~$61). Higher-tier plans correlate with higher churn — possibly because customers on expensive plans feel the price isn't justified.

In [ ]:
# 4.2 Tenure bins — churn rate across early/mid/late lifecycle
df['tenure_band'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0–12 months', '13–24 months', '25–48 months', '49–72 months'],
    include_lowest=True
)

cr_tenure = df.groupby('tenure_band', observed=True)['churn'].mean().mul(100).reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(cr_tenure['tenure_band'].astype(str), cr_tenure['churn'],
              color=[BAR_COLOR, '#E07B3A', '#5C9E6E', BASE_COLOR],
              edgecolor='white', width=0.55, alpha=0.9)
ax.axhline(df['churn'].mean() * 100, color='black', linestyle='--', linewidth=1.2, label='Overall avg (26.5%)')
ax.set_title('Churn Rate by Tenure Band')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 60)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend()
annotate_bars(ax, offset=0.8)
plt.tight_layout()
plt.show()

**Caption:** Churn is highest in the first year (~47%) and drops sharply with tenure. Customers who survive past 2 years churn at near-baseline rates. The first 12 months is the critical retention window.

In [ ]:
# 4.3 Contract type — churn rate bar + count context
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Churn rate
cr_contract = churn_rate('contract').sort_values('churn_rate', ascending=False)
axes[0].bar(cr_contract['contract'], cr_contract['churn_rate'],
            color=BAR_COLOR, edgecolor='white', width=0.5, alpha=0.88)
axes[0].set_title('Churn Rate by Contract Type')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 55)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
annotate_bars(axes[0], offset=0.6)

# Count breakdown
ct = df.groupby(['contract', 'churn']).size().unstack()
ct.rename(columns={0: 'Retained', 1: 'Churned'}).plot(
    kind='bar', ax=axes[1],
    color=[BASE_COLOR, BAR_COLOR], edgecolor='white', width=0.55
)
axes[1].set_title('Churned vs Retained by Contract Type')
axes[1].set_ylabel('Number of Customers')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=10)
axes[1].legend()

fig.suptitle('Contract Type and Churn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Caption:** Month-to-month customers churn at ~43% — more than 8x the rate of two-year contract holders (~3%). This is the single strongest categorical predictor of churn. The volume chart shows that most of the customer base is on month-to-month contracts, making this segment both the highest-risk and the largest.

In [ ]:
# 4.4 Monthly charges distribution — KDE by churn status
fig, ax = plt.subplots(figsize=(10, 4))

for label, color, name in [(0, BASE_COLOR, 'No Churn'), (1, BAR_COLOR, 'Churned')]:
    subset = df[df['churn'] == label]['monthly_charges']
    subset.plot.kde(ax=ax, color=color, linewidth=2.5, label=f'{name} (n={len(subset):,})')
    ax.axvline(subset.mean(), color=color, linestyle='--', linewidth=1.2,
               label=f'{name} mean: ${subset.mean():.0f}')

ax.set_title('Monthly Charges Distribution — Churned vs Retained')
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

**Caption:** Churned customers cluster in the $65–$100 range. Retained customers are more spread out, with a notable peak in the $20–$30 range (basic/no-internet plans). High monthly charges are associated with higher churn risk.

---
## Section 5 — Top 5 Variables Most Influencing Churn

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Encode all categoricals for RF importance
df_enc = df.drop(columns=['tenure_band']).copy()
le = LabelEncoder()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))

X = df_enc.drop(columns=['churn'])
y = df_enc['churn']

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X, y)

importance = pd.Series(rf.feature_importances_, index=X.columns)\
               .sort_values(ascending=False)

print('Feature Importances (Random Forest):')
print(importance.round(4).to_string())

In [ ]:
top5 = importance.head(5)

fig, ax = plt.subplots(figsize=(9, 4.5))
colors = [BAR_COLOR if i < 5 else BASE_COLOR for i in range(len(importance))]
bar_colors_top = ['#C0392B', '#E74C3C', '#E07B3A', '#D4AC0D', '#5C9E6E']

bars = ax.barh(top5.index[::-1], top5.values[::-1],
               color=bar_colors_top[::-1], edgecolor='white', height=0.55)

for bar, val in zip(bars, top5.values[::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=10, fontweight='bold')

ax.set_title('Top 5 Features Influencing Churn (Random Forest Importance)', pad=12)
ax.set_xlabel('Feature Importance Score')
ax.set_xlim(0, top5.max() * 1.25)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('\nTop 5:')
for rank, (feat, score) in enumerate(top5.items(), 1):
    print(f'  {rank}. {feat}: {score:.4f}')

In [ ]:
# Full importance ranked bar — all features
fig, ax = plt.subplots(figsize=(10, 7))
c = [BAR_COLOR if f in top5.index else '#AABDCC' for f in importance.index[::-1]]
ax.barh(importance.index[::-1], importance.values[::-1],
        color=c[::-1][::-1], edgecolor='white', height=0.65)
ax.set_title('All Feature Importances (highlighted: top 5)', pad=12)
ax.set_xlabel('Importance Score')
ax.invert_xaxis()
plt.tight_layout()
plt.show()

**Caption:** The Random Forest confirms what the visual EDA suggested. The top 5 variables are:

| Rank | Feature | Why It Matters |
|------|---------|----------------|
| 1 | **total_charges** | Proxy for customer lifetime value and accumulated relationship — longer-tenured customers have higher totals and churn less |
| 2 | **tenure** | Most powerful raw signal — early-lifecycle customers are far more likely to leave |
| 3 | **monthly_charges** | Higher bills = higher churn risk; expensive plans aren't delivering perceived value |
| 4 | **contract** | Month-to-month = no switching cost; two-year contracts lock in loyalty |
| 5 | **internet_service** | Fiber optic users churn at 2x the DSL rate; service quality perception issue |

Note: `total_charges` and `tenure` are correlated, so their combined importance is somewhat shared — but both independently carry signal beyond each other.

---
## Section 6 — Cross-Variable Deep Dives

In [ ]:
# 6.1 Contract × Tenure by churn — violin
fig, ax = plt.subplots(figsize=(11, 5))
sns.violinplot(
    data=df, x='contract', y='tenure',
    hue='churn', split=True, inner='quartile',
    palette={0: BASE_COLOR, 1: BAR_COLOR},
    ax=ax
)
ax.set_title('Tenure Distribution by Contract Type and Churn Status')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Tenure (months)')
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles, ['Retained', 'Churned'], title='Churn')
plt.tight_layout()
plt.show()

**Caption:** Month-to-month churners are concentrated at very low tenure (0–20 months). For one- and two-year contracts, the churn volume is so small that the violin barely registers on the churned side — confirming contract type as a near-complete churn insulator for long-term holders.

In [ ]:
# 6.2 Payment method × churn
fig, ax = plt.subplots(figsize=(10, 4))
cr_pay = churn_rate('payment_method').sort_values('churn_rate', ascending=False)
bar_c = [BAR_COLOR if v > 26 else BASE_COLOR for v in cr_pay['churn_rate']]
ax.bar(cr_pay['payment_method'], cr_pay['churn_rate'],
       color=bar_c, edgecolor='white', width=0.55, alpha=0.9)
ax.set_title('Churn Rate by Payment Method')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 55)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.axhline(df['churn'].mean() * 100, color='black', linestyle='--', linewidth=1)
annotate_bars(ax, offset=0.6)
ax.tick_params(axis='x', rotation=10)
plt.tight_layout()
plt.show()

**Caption:** Electronic check users churn at ~45% — nearly 3x the rate of automatic payment methods (~15–18%). Automatic payments (bank transfer, credit card) create passive retention through friction reduction. Electronic/mailed check users require active re-engagement to pay, making exit easier.

In [ ]:
# 6.3 Paperless billing × churn
fig, ax = plt.subplots(figsize=(5, 4))
cr_pb = churn_rate('paperless_billing')
ax.bar(cr_pb['paperless_billing'], cr_pb['churn_rate'],
       color=[BAR_COLOR, BASE_COLOR], edgecolor='white', width=0.4)
ax.set_title('Churn Rate by Paperless Billing')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 45)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
annotate_bars(ax, offset=0.5)
plt.tight_layout()
plt.show()

**Caption:** Paperless billing customers churn at ~34% vs ~17% for paper billing. This is likely a confound — paperless billing users skew toward the tech-savvy, younger demographic on fiber optic / month-to-month contracts, which are already high-churn segments.

---
## Section 7 — Three Key Business Insights

### Insight 1: Month-to-month contracts are a ticking clock
Customers on month-to-month plans churn at **43%**, compared to **11%** on one-year and **3%** on two-year contracts. Month-to-month is also the most popular contract type, meaning the majority of the customer base is in the highest-risk bucket. **Business impact:** A targeted incentive campaign offering discounts or loyalty rewards to convert month-to-month customers to annual contracts could directly reduce overall churn by 10–15 percentage points.

### Insight 2: Fiber optic customers are unhappy — and they're the high-value segment
Fiber optic users churn at **~42%** despite paying the highest monthly charges. These are the most valuable customers by revenue, and they're leaving at the highest rate. The churn is not about price sensitivity (they're already paying more) — it points to **service quality or expectation mismatch**. **Business impact:** Improving fiber optic reliability, onboarding experience, or bundling Tech Support into fiber plans could retain the highest-LTV customers in the base.

### Insight 3: The first 12 months are make-or-break
Churn rate in the first year is **~47%**, dropping to ~35% in year 2 and below average by year 3. Customers who stay past 2 years almost never leave. **Business impact:** Concentrating retention efforts (onboarding calls, loyalty perks, proactive support outreach) into the first 12 months would have the highest ROI — retaining a customer through their first year essentially retains them long-term.

---
## Section 8 — Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Feature': 'contract',         'Segment with Highest Churn': 'Month-to-month', 'Churn Rate': '~43%', 'Recommended Action': 'Incentivize annual contract upgrades'},
    {'Feature': 'internet_service', 'Segment with Highest Churn': 'Fiber optic',    'Churn Rate': '~42%', 'Recommended Action': 'Improve fiber QoS; bundle support'},
    {'Feature': 'tenure_band',      'Segment with Highest Churn': '0–12 months',    'Churn Rate': '~47%', 'Recommended Action': 'Strengthen early onboarding & outreach'},
    {'Feature': 'tech_support',     'Segment with Highest Churn': 'No',             'Churn Rate': '~42%', 'Recommended Action': 'Promote Tech Support add-on adoption'},
    {'Feature': 'online_security',  'Segment with Highest Churn': 'No',             'Churn Rate': '~42%', 'Recommended Action': 'Bundle security in entry plans'},
    {'Feature': 'senior_citizen',   'Segment with Highest Churn': 'Yes',            'Churn Rate': '~41%', 'Recommended Action': 'Senior-specific loyalty program'},
    {'Feature': 'payment_method',   'Segment with Highest Churn': 'Electronic check','Churn Rate': '~45%','Recommended Action': 'Nudge toward automatic payment setup'},
    {'Feature': 'dependents',       'Segment with Highest Churn': 'No',             'Churn Rate': '~31%', 'Recommended Action': 'Family plan promotions to anchor customers'},
])

print('=== Churn Risk Summary by Feature ===')
summary